# Sample Cypher Queries

Reusable queries that work on **any** graph.

**Prerequisite:** Run the IPL or Datacenter notebook first.

In [ ]:
import requests, pandas as pd

CP = 'http://control-plane:8080'
HEADERS = {'Content-Type': 'application/json', 'X-Username': 'demo@example.com', 'X-User-Role': 'admin'}

# Find running instances
instances = requests.get(f'{CP}/api/instances', headers=HEADERS).json().get('data', [])
running = [i for i in instances if i['status'] == 'running']

if not running:
    print('No running instances! Run the IPL or Datacenter notebook first.')
else:
    for i in running:
        print(f"  {i['id']}: {i['name']} ({i['wrapper_type']})")
    # Use first running instance
    inst = running[0]
    W = f'http://{inst["pod_name"]}:8000'
    print(f'\nUsing: {inst["name"]}')

In [ ]:
def q(cypher):
    r = requests.post(f'{W}/query', json={'query': cypher})
    d = r.json()
    if 'error' in d: print(f'Error: {d["error"]}'); return pd.DataFrame()
    return pd.DataFrame(d['rows'], columns=d['columns'])

In [ ]:
# Node types and counts
q('MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count ORDER BY count DESC')

In [ ]:
# Relationship types
q('MATCH ()-[r]->() RETURN type(r) AS relationship, count(r) AS count ORDER BY count DESC')

In [ ]:
# Schema
q('MATCH (a)-[r]->(b) RETURN DISTINCT labels(a)[0] AS from_type, type(r) AS rel, labels(b)[0] AS to_type')

In [ ]:
# Most connected nodes
q('MATCH (n)-[r]-() RETURN labels(n)[0] AS type, coalesce(n.name, n.hostname, id(n)) AS name, count(r) AS connections ORDER BY connections DESC LIMIT 15')

In [ ]:
# 2-hop paths
q('MATCH (a)-[r1]->(b)-[r2]->(c) RETURN labels(a)[0] AS start, type(r1) AS rel1, labels(b)[0] AS mid, type(r2) AS rel2, labels(c)[0] AS end LIMIT 20')

In [ ]:
# To switch instance, change the index:
# inst = running[1]
# W = f'http://{inst["pod_name"]}:8000'
# print(f'Switched to: {inst["name"]}')